# AFCNet: Clean Reproducibility Notebook

This notebook provides a compact, reproducibility-oriented workflow for the **vehicle-counting branch** of AFCNet: dataset inspection, deterministic image/annotation pairing, model construction, training, held-out evaluation, checkpointing, and complexity/latency measurement.

## Scope and interpretation

- The counting architecture follows the intended design in the supplied notebook: MobileNetV3-Large, CBAM attention, global–local feature gating, and a residual regression head.
- Two shape/implementation errors in the original model cell are corrected here so the model can execute: CBAM is inserted at feature stages with matching channel counts, spatial attention multiplies the original feature tensor, and the residual path is projected to the matching dimension.
- The original notebook did not establish an independent held-out test result. This clean notebook uses **train/validation/test partitions** so evaluation can be performed on images not used for gradient updates. If images are sequential frames from the same video, use video-level splits to avoid adjacent-frame leakage; a simple frame-level split is not sufficient for a rigorous sequence-level generalization claim.
- The original notebook's scene-change F1 calculation used mock labels, and its scene feature extractor was untrained. Therefore this notebook does **not** report scene-change accuracy/F1 or claim a trained SFENet/TSC branch. It includes an optional, explicitly unsupervised SSIM-based change-score demonstration only.
- No performance numbers are hard-coded. All metrics must be generated by running the notebook on the specified data and hardware.

## Expected annotation format

This notebook expects image files and same-stem `.txt` annotation files, for example `frame001.jpg` and `frame001.txt`. The target count is the number of **non-empty annotation lines** in each text file. This is appropriate only if each line represents one vehicle. If your annotation files use another format, adapt the target-count parser before training.

## Suggested run order

Run cells from top to bottom. Set the dataset paths in the configuration cell before running dataset inspection.

## 1. Install dependencies

Run this cell in a fresh Colab or Python environment. On Colab, PyTorch and torchvision are usually preinstalled; this cell only installs the additional libraries used below.

In [ ]:
%pip -q install scikit-learn pandas matplotlib pillow scikit-image

## 2. Imports and reproducibility settings

In [ ]:
import os
import json
import time
import random
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Configure dataset paths and experiment settings

Set `IMAGE_DIR` and `LABEL_DIR` to folders containing the image files and matching `.txt` annotations. Recursive discovery is enabled. Keep the selected data and split protocol fixed when comparing experiments.

In [ ]:
# EDIT THESE TWO PATHS for your environment.
# Example Colab paths:
IMAGE_DIR = Path("/content/ua_detrac/images")
LABEL_DIR = Path("/content/ua_detrac/labels")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
MAX_SAMPLES = None          # e.g. 20000 for a fixed subset; None uses all matched pairs
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
BATCH_SIZE = 8
EPOCHS = 20
LEARNING_RATE = 1e-4
NUM_WORKERS = 2
IMAGE_SIZE = 224
USE_PRETRAINED = True
OUTPUT_DIR = Path("outputs/afcnet")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert abs(TRAIN_FRACTION + VAL_FRACTION + TEST_FRACTION - 1.0) < 1e-8
print("Image directory exists:", IMAGE_DIR.exists(), IMAGE_DIR)
print("Label directory exists:", LABEL_DIR.exists(), LABEL_DIR)

## 4. Pair images with annotations and inspect the dataset

Pairing is performed by relative path (when possible) and then by unique filename stem. Ambiguous duplicate stems are skipped and reported rather than silently paired by sorted list position. The count target is the number of non-empty lines in the matched annotation file.

In [ ]:
def build_pairs(image_dir: Path, label_dir: Path):
    if not image_dir.exists() or not label_dir.exists():
        raise FileNotFoundError(
            f"Set IMAGE_DIR and LABEL_DIR correctly. Got {image_dir} and {label_dir}"
        )

    images = sorted(p for p in image_dir.rglob("*")
                    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    labels = sorted(label_dir.rglob("*.txt"))

    # Prefer relative paths when directory structures mirror one another.
    label_by_rel = {}
    for p in labels:
        rel = p.relative_to(label_dir).with_suffix("")
        label_by_rel.setdefault(str(rel), []).append(p)

    labels_by_stem = {}
    for p in labels:
        labels_by_stem.setdefault(p.stem, []).append(p)

    pairs, missing, ambiguous = [], [], []
    for image_path in images:
        rel_key = str(image_path.relative_to(image_dir).with_suffix(""))
        candidates = label_by_rel.get(rel_key, [])
        if len(candidates) == 1:
            label_path = candidates[0]
        elif len(candidates) > 1:
            ambiguous.append(str(image_path))
            continue
        else:
            candidates = labels_by_stem.get(image_path.stem, [])
            if len(candidates) == 1:
                label_path = candidates[0]
            elif len(candidates) > 1:
                ambiguous.append(str(image_path))
                continue
            else:
                missing.append(str(image_path))
                continue

        try:
            text = label_path.read_text(encoding="utf-8", errors="ignore")
            count = sum(1 for line in text.splitlines() if line.strip())
        except OSError:
            missing.append(str(image_path))
            continue
        pairs.append({"image": str(image_path), "label": str(label_path), "count": count})

    report = {
        "images_found": len(images),
        "label_files_found": len(labels),
        "matched_pairs": len(pairs),
        "images_without_unambiguous_label": len(missing),
        "ambiguous_image_matches": len(ambiguous),
    }
    return pairs, report, missing, ambiguous

pairs, pairing_report, missing_images, ambiguous_images = build_pairs(IMAGE_DIR, LABEL_DIR)
print(json.dumps(pairing_report, indent=2))
if MAX_SAMPLES is not None:
    pairs = pairs[:MAX_SAMPLES]
print("Pairs selected:", len(pairs))
if not pairs:
    raise RuntimeError("No matched image/label pairs. Check paths and annotation format.")

manifest = pd.DataFrame(pairs)
display(manifest.head())
display(manifest["count"].describe().to_frame("vehicle_count"))

## 5. Create reproducible train/validation/test splits

The default split is randomized with a fixed seed. **For video-sequence data, replace this with a video-level split** using the sequence/video identifier, so frames from the same video do not leak across partitions. The notebook does not infer video IDs automatically because folder naming conventions vary by dataset export.

In [ ]:
indices = np.arange(len(pairs))
train_idx, holdout_idx = train_test_split(
    indices, test_size=(VAL_FRACTION + TEST_FRACTION), random_state=SEED, shuffle=True
)
relative_test = TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=relative_test, random_state=SEED, shuffle=True
)

split_manifest = manifest.copy()
split_manifest["split"] = "train"
split_manifest.loc[val_idx, "split"] = "validation"
split_manifest.loc[test_idx, "split"] = "test"
split_manifest.to_csv(OUTPUT_DIR / "data_split_manifest.csv", index=False)

print(split_manifest["split"].value_counts())
display(split_manifest.groupby("split")["count"].agg(["count", "mean", "std", "min", "max"]))

## 6. Dataset, transforms, and data loaders

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

class VehicleCountDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        with Image.open(record["image"]) as image:
            image = image.convert("RGB")
            if self.transform:
                image = self.transform(image)
        target = torch.tensor(float(record["count"]), dtype=torch.float32)
        return image, target, record["image"]

train_records = [pairs[i] for i in train_idx]
val_records = [pairs[i] for i in val_idx]
test_records = [pairs[i] for i in test_idx]

train_loader = DataLoader(
    VehicleCountDataset(train_records, train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)
val_loader = DataLoader(
    VehicleCountDataset(val_records, eval_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)
test_loader = DataLoader(
    VehicleCountDataset(test_records, eval_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda")
)

print("Train / validation / test:", len(train_records), len(val_records), len(test_records))

## 7. AFCNet counting architecture

This cell implements the intended counting architecture in one place. The code corrects execution-blocking tensor-shape issues found in the original notebook. These corrections should be disclosed in the GitHub README and manuscript's reproducibility statement.

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(1, channels // reduction)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = self.avg_pool(x).flatten(1)
        maximum = self.max_pool(x).flatten(1)
        weight = self.sigmoid(self.mlp(avg) + self.mlp(maximum))
        return x * weight[:, :, None, None]


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        maximum = torch.max(x, dim=1, keepdim=True).values
        attention = torch.sigmoid(self.conv(torch.cat([avg, maximum], dim=1)))
        return x * attention


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel = ChannelAttention(channels, reduction)
        self.spatial = SpatialAttention()

    def forward(self, x):
        return self.spatial(self.channel(x))


class GlobalLocalFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(channels * 2, channels),
            nn.ReLU(inplace=True),
            nn.Linear(channels, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        avg = x.mean(dim=(2, 3))
        maximum = torch.amax(x, dim=(2, 3))
        weights = self.gate(torch.cat([avg, maximum], dim=1))
        return x * weights[:, :, None, None]


class ResidualDenseRegressionHead(nn.Module):
    def __init__(self, in_features=960, dropout=0.4):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 512)
        self.fc2 = nn.Linear(512, 256)
        self.residual_projection = nn.Linear(512, 256, bias=False)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 1)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = self.relu(self.fc1(x))
        out = self.relu(self.fc2(self.dropout(residual)))
        out = out + 0.3 * self.residual_projection(residual)
        out = self.relu(self.fc3(self.dropout(out)))
        return self.fc4(out)


class AFCNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
        backbone = models.mobilenet_v3_large(weights=weights)
        self.features = backbone.features
        # MobileNetV3-Large stages 4 and 12 have 40 and 112 channels respectively.
        self.cbam_40 = CBAM(40)
        self.cbam_112 = CBAM(112)
        self.cbam_960 = CBAM(960)
        self.global_local_fusion = GlobalLocalFusion(960)
        self.regression_head = ResidualDenseRegressionHead(960)

    def forward(self, x):
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i == 4:
                x = self.cbam_40(x)
            elif i == 12:
                x = self.cbam_112(x)
        x = self.cbam_960(x)
        x = self.global_local_fusion(x)
        x = nn.functional.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.regression_head(x).squeeze(1)

model = AFCNet(pretrained=USE_PRETRAINED).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model.__class__.__name__, "| trainable parameters:", f"{n_params:,}")

## 8. Train with validation-based checkpoint selection

The optimizer and loss follow the notebook's primary counting setup (Adam, learning rate 1e-4, MSE loss, 20 epochs by default). Unlike the original training cell, this version evaluates the validation partition after each epoch and saves the checkpoint with the lowest validation MAE.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, n_batches = 0.0, 0
    all_targets, all_preds, all_paths = [], [], []

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, targets, paths in loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            predictions = model(images)
            loss = criterion(predictions, targets)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += float(loss.detach().cpu())
            n_batches += 1
            all_targets.extend(targets.detach().cpu().numpy().tolist())
            all_preds.extend(predictions.detach().cpu().numpy().tolist())
            all_paths.extend(list(paths))

    y_true = np.asarray(all_targets, dtype=np.float64)
    y_pred = np.asarray(all_preds, dtype=np.float64)
    metrics = {
        "mean_batch_mse": total_loss / max(1, n_batches),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "n_samples": int(len(y_true)),
    }
    return metrics, y_true, y_pred, all_paths

history = []
best_val_mae = float("inf")
best_path = OUTPUT_DIR / "afcnet_best.pt"

for epoch in range(1, EPOCHS + 1):
    train_metrics, _, _, _ = run_epoch(model, train_loader, optimizer)
    val_metrics, _, _, _ = run_epoch(model, val_loader)
    record = {"epoch": epoch, "train": train_metrics, "validation": val_metrics}
    history.append(record)
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train MAE={train_metrics['mae']:.4f}, RMSE={train_metrics['rmse']:.4f} | "
        f"val MAE={val_metrics['mae']:.4f}, RMSE={val_metrics['rmse']:.4f}"
    )

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": {
            "seed": SEED, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
            "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
            "pretrained": USE_PRETRAINED, "train_fraction": TRAIN_FRACTION,
            "validation_fraction": VAL_FRACTION, "test_fraction": TEST_FRACTION,
        },
        "train_metrics": train_metrics,
        "validation_metrics": val_metrics,
    }
    torch.save(checkpoint, OUTPUT_DIR / "afcnet_last.pt")
    if val_metrics["mae"] < best_val_mae:
        best_val_mae = val_metrics["mae"]
        torch.save(checkpoint, best_path)

pd.DataFrame([
    {"epoch": h["epoch"], **{f"train_{k}":v for k,v in h["train"].items()},
     **{f"val_{k}":v for k,v in h["validation"].items()}}
    for h in history
]).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
(OUTPUT_DIR / "run_config.json").write_text(json.dumps({
    "seed": SEED, "device": str(DEVICE), "python": platform.python_version(),
    "torch": torch.__version__, "image_dir": str(IMAGE_DIR), "label_dir": str(LABEL_DIR),
    "pairing_report": pairing_report, "n_samples": len(pairs),
    "n_train": len(train_records), "n_validation": len(val_records), "n_test": len(test_records),
    "trainable_parameters": n_params,
}, indent=2))

## 9. Evaluate once on the held-out test set

The test partition is not used for gradient updates or checkpoint selection. Run this cell after training. Do not repeatedly tune model choices based on test-set results.

In [ ]:
checkpoint = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
test_metrics, y_test, p_test, test_paths = run_epoch(model, test_loader)
test_results = pd.DataFrame({
    "image": test_paths,
    "ground_truth_count": y_test,
    "predicted_count_raw": p_test,
    "predicted_count_nonnegative": np.maximum(p_test, 0.0),
})
test_results.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

print("Held-out test metrics (raw regression predictions)")
print(json.dumps(test_metrics, indent=2))
display(test_results.head())

plt.figure(figsize=(6, 6))
plt.scatter(y_test, p_test, alpha=0.5)
lo = min(float(np.min(y_test)), float(np.min(p_test)))
hi = max(float(np.max(y_test)), float(np.max(p_test)))
plt.plot([lo, hi], [lo, hi], linestyle="--")
plt.xlabel("Ground-truth vehicle count")
plt.ylabel("Predicted vehicle count")
plt.title("AFCNet: held-out test predictions")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "test_prediction_scatter.png", dpi=300)
plt.show()

## 10. Model complexity and inference timing

The parameter count is measured directly from the instantiated model. FLOPs/MACs depend on the counting convention and tool used; this cell reports parameter count and empirical latency/throughput without inventing FLOPs. Timing uses the held-out test loader and includes batch processing, not the entire data-loading/visualization pipeline. Results depend on hardware, batch size, warm-up, and software versions.

In [ ]:
model.eval()
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Warm-up and measure forward-pass time on batches already loaded by the DataLoader.
batch_times, batch_sizes = [], []
with torch.no_grad():
    for batch_i, (images, targets, paths) in enumerate(test_loader):
        images = images.to(DEVICE)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = model(images)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        if batch_i >= 2:  # discard first two batches as warm-up
            batch_times.append(elapsed)
            batch_sizes.append(len(images))

if batch_times:
    total_images = sum(batch_sizes)
    total_seconds = sum(batch_times)
    timing = {
        "trainable_parameters": int(param_count),
        "measured_batches": len(batch_times),
        "batch_size_setting": BATCH_SIZE,
        "mean_batch_latency_ms": float(np.mean(batch_times) * 1000),
        "mean_image_latency_ms_approx": float(total_seconds / total_images * 1000),
        "throughput_images_per_second": float(total_images / total_seconds),
        "device": str(DEVICE),
    }
    print(json.dumps(timing, indent=2))
    (OUTPUT_DIR / "complexity_timing.json").write_text(json.dumps(timing, indent=2))
else:
    print("Not enough test batches for timing; increase test-set size or reduce batch size.")

## 11. Optional exploratory scene-change analysis using SSIM

This section is deliberately separate from the counting model. It calculates frame-to-frame SSIM dissimilarity for images in the order supplied by the user. It is meaningful only if `SCENE_FRAME_DIR` contains frames from a single sequence in chronological filename order. The threshold is mean + 2 standard deviations over this sequence and is a heuristic, not a learned model. Without genuine manually annotated change points, do not report accuracy, precision, recall, or F1.

In [ ]:
from skimage.metrics import structural_similarity

# Set this to one video sequence extracted into chronologically named image files.
SCENE_FRAME_DIR = None  # e.g. Path("/content/ua_detrac/one_video_sequence")

def ssim_change_scores(frame_dir):
    frame_dir = Path(frame_dir)
    paths = sorted(p for p in frame_dir.iterdir()
                   if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if len(paths) < 2:
        raise ValueError("Provide at least two chronologically ordered frames.")
    scores = []
    for prev_path, curr_path in zip(paths[:-1], paths[1:]):
        with Image.open(prev_path) as im:
            prev = np.asarray(im.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE)))
        with Image.open(curr_path) as im:
            curr = np.asarray(im.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE)))
        score = structural_similarity(prev, curr, channel_axis=2, data_range=255)
        scores.append({"previous_frame": str(prev_path), "current_frame": str(curr_path),
                       "ssim_dissimilarity": float(1.0 - score)})
    frame_scores = pd.DataFrame(scores)
    threshold = frame_scores["ssim_dissimilarity"].mean() + 2 * frame_scores["ssim_dissimilarity"].std(ddof=0)
    frame_scores["heuristic_change"] = frame_scores["ssim_dissimilarity"] > threshold
    return frame_scores, float(threshold)

if SCENE_FRAME_DIR is not None:
    scene_scores, scene_threshold = ssim_change_scores(SCENE_FRAME_DIR)
    scene_scores.to_csv(OUTPUT_DIR / "ssim_scene_change_scores.csv", index=False)
    print("Heuristic threshold:", scene_threshold)
    print("Candidate transitions:", int(scene_scores["heuristic_change"].sum()))
    display(scene_scores.head())
else:
    print("Skipped: set SCENE_FRAME_DIR to a single chronological frame sequence to run this optional analysis.")

## 12. Output files and reporting checklist

The run creates files under `outputs/afcnet/`:

- `data_split_manifest.csv` — exact image/label pairs and split assignments.
- `run_config.json` — paths, versions, split counts, seed, and model parameter count.
- `training_history.csv` — per-epoch training and validation metrics.
- `afcnet_best.pt` and `afcnet_last.pt` — best-validation and final checkpoints.
- `test_predictions.csv` — held-out test predictions.
- `test_prediction_scatter.png` — ground-truth versus predicted counts.
- `complexity_timing.json` — measured parameter count, latency, and throughput.
- `ssim_scene_change_scores.csv` — created only if the optional scene-change section is run.

### Reproducibility checklist

Before publishing numerical results, record the dataset version/source, exact split strategy, annotation-count convention, software versions, GPU model, pretrained-weight setting, seed, and all measured metrics. For video data, prefer video-level train/validation/test partitions. Do not treat count-derived severity labels as independent classification ground truth, and do not report scene-change F1 without genuine annotated change points.

This notebook does not claim that the original notebook's previously reported metrics have been reproduced. The values reported in a paper should be taken from a complete run of this notebook (or a clearly specified equivalent protocol) and checked against saved outputs.